### Sentiment Analysis - using RNN
- Obviously long range dependecy and parallelization issues will be there
- The eobjective is to implement the same on a small dataset

In [1]:
import os
import pandas as pd
import torch
import numpy as np

from torch import nn
from sklearn.model_selection import train_test_split

data = pd.read_csv('data/sentiment_dataset.csv')

In [2]:
data

,text,sentiment,topic
0,Agent every development say quality throughout...,2,Tech
1,All behavior discussion own night respond red ...,0,Tech
2,Recently future choice whatever from behavior ...,0,Tech
3,Enough analysis least by two bad fall pick tho...,2,Product
4,World talk term herself law street class great...,0,Product
...,...,...,...
495,Name door production back us off main option l...,0,Tech
496,Computer what attorney board sport side her ma...,0,Service
497,Tough player a chair chance most hope official...,1,Product
498,Truth view response trip wrong old particularl...,2,Service


In [7]:
X = data["text"]
y = data["sentiment"]

X_train,X_test,y_train,y_test = train_test_split(X, y, train_size=0.80, random_state = 15, stratify = y)

In [9]:
train_mask = y_train != 2
test_mask = y_test != 2

X_train = X_train[train_mask]
y_train = y_train[train_mask]

X_test = X_test[test_mask]
y_test = y_test[test_mask]

In [10]:
set(data['sentiment'])

{0, 1, 2}

#### RNN definition

In [11]:
class CustomRNN(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(CustomRNN, self).__init__()
        
        self.hidden_dim = hidden_dim
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        self.ih = nn.Linear(embedding_dim, hidden_dim) #input

        self.hh = nn.Linear(hidden_dim, hidden_dim) #hidden - hidden

        self.tanh = nn.Tanh() #tanmh activation

        self.fc = nn.Linear(hidden_dim, output_dim) #Output layer to map final hidden state to sentiment scores

    def forward(self, text): # text shape: [batch_size, sequence_length]
        
        embedded = self.embedding(text) # embedded shape: [batch_size, sequence_length, embedding_dim]

        #Permute to iterate over the time dimension (sequence length)
        embedded = embedded.permute(1, 0, 2) #[sequence_length, batch_size, embedding_dim]

        batch_size = text.size(0)

        hidden = torch.zeros(batch_size, self.hidden_dim, device=text.device) # hidden shape: [batch_size, hidden_dim]

        for t in range(embedded.size(0)): #step using one token at a time
            
            x_t = embedded[t]  # Input vector at time t: [batch_size, embedding_dim]

            hidden = self.tanh(self.ih(x_t) + self.hh(hidden)) # Calculate the next hidden state

        output = self.fc(hidden) # output shape: [batch_size, output_dim]

        return output

#### Text Preprocessing
- lowercase text
- tokenize
- build vocabulary
- reserve indices for <PAD> and <UNK>

In [12]:
from collections import Counter

# Vocab
def tokenize(text):
    return text.lower().split()


counter = Counter()

for sentence in X_train:
    counter.update(tokenize(sentence))


vocab = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, _ in counter.items():
    vocab[word] = len(vocab)


print(f"Vocabulary Size : {len(vocab)}")

Vocabulary Size : 1084


In [13]:
list(enumerate(vocab))[:5]

[(0, '<PAD>'), (1, '<UNK>'), (2, 'accept'), (3, 'result'), (4, 'as')]

#### ENcode and pad the seq

In [14]:
def encode_sentence(sentence, vocab):
    tokens = tokenize(sentence)
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]


X_train_encoded = [encode_sentence(sentence, vocab) for sentence in X_train]
X_test_encoded  = [encode_sentence(sentence, vocab) for sentence in X_test]

In [15]:
max_length = max(len(sentence) for sentence in X_train_encoded)

print(f"Maximum Seq Length : {max_length}")

Maximum Seq Length : 13


In [16]:
def pad_sequence(sequence, max_length, pad_value=0):
    if len(sequence) < max_length:
        sequence = sequence + [pad_value] * (max_length - len(sequence))
    else:
        sequence = sequence[:max_length]

    return sequence

Need torch data loader

In [17]:
X_train_padded = [
    pad_sequence(sentence, max_length)
    for sentence in X_train_encoded
]

X_test_padded = [
    pad_sequence(sentence, max_length)
    for sentence in X_test_encoded
]

In [18]:
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor  = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([252, 13])
torch.Size([252])


#### Batching is needed, so do the dataloader

In [19]:
from torch.utils.data import Dataset, DataLoader

class SentimentDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [20]:
train_dataset = SentimentDataset(
    X_train_tensor,
    y_train_tensor
)

test_dataset = SentimentDataset(
    X_test_tensor,
    y_test_tensor
)

In [21]:
batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [22]:
X_batch, y_batch = next(iter(train_loader))

print("Input Shape :", X_batch.shape)
print("Labels Shape:", y_batch.shape)

Input Shape : torch.Size([64, 13])
Labels Shape: torch.Size([64])


#### Model initialization

In [28]:
vocab_size = len(vocab)
embedding_dim = 100
hidden_size = 128
output_size = 2

learning_rate = 0.001
num_epochs = 100

In [29]:
model = CustomRNN(
    vocab_size=vocab_size,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_size,
    output_dim=output_size
)

print(model)
#def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim)

CustomRNN(
  (embedding): Embedding(1084, 100)
  (ih): Linear(in_features=100, out_features=128, bias=True)
  (hh): Linear(in_features=128, out_features=128, bias=True)
  (tanh): Tanh()
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [30]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=learning_rate)

In [31]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable Parameters: {total_params:,}")

Trainable Parameters: 138,098


#### Training loop

In [32]:
for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    for X_batch, y_batch in train_loader:

        outputs = model(X_batch) # Forward Pass

        loss = criterion(outputs, y_batch) # Compute Loss

        optimizer.zero_grad() # Clear Previous Gradients

        loss.backward() # Backpropagation

        optimizer.step() # Backpropagation

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Loss: {epoch_loss:.4f}"
    )

Epoch [1/100] Loss: 0.7166
Epoch [2/100] Loss: 0.6803
Epoch [3/100] Loss: 0.6479
Epoch [4/100] Loss: 0.6325
Epoch [5/100] Loss: 0.6063
Epoch [6/100] Loss: 0.5789
Epoch [7/100] Loss: 0.5416
Epoch [8/100] Loss: 0.4938
Epoch [9/100] Loss: 0.4364
Epoch [10/100] Loss: 0.3641
Epoch [11/100] Loss: 0.2895
Epoch [12/100] Loss: 0.2083
Epoch [13/100] Loss: 0.1404
Epoch [14/100] Loss: 0.0750
Epoch [15/100] Loss: 0.0417
Epoch [16/100] Loss: 0.0276
Epoch [17/100] Loss: 0.0143
Epoch [18/100] Loss: 0.0081
Epoch [19/100] Loss: 0.0062
Epoch [20/100] Loss: 0.0044
Epoch [21/100] Loss: 0.0029
Epoch [22/100] Loss: 0.0020
Epoch [23/100] Loss: 0.0016
Epoch [24/100] Loss: 0.0014
Epoch [25/100] Loss: 0.0013
Epoch [26/100] Loss: 0.0012
Epoch [27/100] Loss: 0.0011
Epoch [28/100] Loss: 0.0010
Epoch [29/100] Loss: 0.0009
Epoch [30/100] Loss: 0.0009
Epoch [31/100] Loss: 0.0008
Epoch [32/100] Loss: 0.0008
Epoch [33/100] Loss: 0.0007
Epoch [34/100] Loss: 0.0007
Epoch [35/100] Loss: 0.0007
Epoch [36/100] Loss: 0.0007
E

#### Evaluation on test

In [33]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        outputs = model(X_batch)

        predictions = torch.argmax(outputs, dim=1)

        correct += (predictions == y_batch).sum().item()
        total += y_batch.size(0)

accuracy = 100 * correct / total

print(f"Test Accuracy: {accuracy:.2f}%")

Test Accuracy: 60.32%
